# Affine initialization and B-spline SVF registration

Register two small synthetic 2-D images without downloads. First fit an affine
map, then optimize a B-spline stationary velocity field on top of that map.
Inspect alignment, residual displacements and the total-map Jacobian.

Install ANTsTorch and notebook tools from the repository root before opening this notebook:
```bash
python -m pip install -e '.[notebooks]'
python -m ipykernel install --user --name python3 --display-name "Python 3 (ANTsTorch)"
```
Select that kernel and run the cells in order. All examples default to CPU.

In [ ]:
import torch  # Import before ANTs to avoid OpenMP conflicts on macOS.
import numpy as np
import ants
import antstorch

# Display figures inside Jupyter/VS Code even if MPLBACKEND was set to Agg.
from IPython import get_ipython
shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

device = "cpu"
antstorch.set_default_device(device)

In [ ]:
from antstorch.bspline_flows import (
    ImageDomain, affine_displacement_field, affine_registration,
    bspline_svf_registration, warp_image,
)
torch.manual_seed(42)
domain = ImageDomain((32, 28), spacing=(1., 1.))
yy, xx = torch.meshgrid(torch.linspace(-1, 1, 28, device=device),
                        torch.linspace(-1, 1, 32, device=device), indexing="ij")
moving = (torch.exp(-9*(xx**2 + yy**2))
          + 0.5*torch.exp(-25*((xx-0.4)**2 + (yy+0.3)**2)))[None, None]
matrix = torch.tensor([[1.04, 0.03], [-0.02, 0.97]], device=device)
translation = torch.tensor([1.2, -0.8], device=device)
known_field = affine_displacement_field(matrix, translation, domain, moving)
known_field[:, 0] += 0.5*torch.sin(np.pi*yy)*torch.cos(np.pi*xx)
fixed = warp_image(moving, known_field, domain, padding_mode="border")
affine = affine_registration(
    fixed, moving, domain, transform_type="Affine", similarity="mse",
    iterations=(60, 60), learning_rate=0.03, shrink_factors=(2, 1),
    smoothing_sigmas=(1., 0.), multi_start=False, padding_mode="border",
)
result = bspline_svf_registration(
    fixed, moving, domain, initial_affine=(affine["matrix"], affine["translation"]),
    mesh_size=3, iterations=40, optimizer="adam", learning_rate=0.03,
    similarity="mse", bending_weight=1e-3, squaring_steps=5,
    padding_mode="border",
)

## Check alignment and local volume changes
Tensor axes are `(batch, channels, y, x)`; displacement components are physical
x and y. `fwdtransforms` contains only the residual SVF displacement, while
the affine is returned separately. The Jacobian refers to the total map.
Do not apply the residual field alone and expect the complete registration.

In [ ]:
warped = result["warpedmovout"]
jacobian = result["jacobian_determinant"]
displacement = result["fwdtransforms"]
initial_mse = (fixed-moving).square().mean().item()
affine_mse = (fixed-affine["warpedmovout"]).square().mean().item()
final_mse = (fixed-warped).square().mean().item()
assert warped.shape == fixed.shape
assert torch.isfinite(warped).all() and torch.isfinite(jacobian).all()
assert final_mse < initial_mse
assert (jacobian > 0).all()
print(f"MSE: unregistered={initial_mse:.6f}; affine={affine_mse:.6f}; affine+SVF={final_mse:.6f}")
print("Nonpositive Jacobian voxels:", int((jacobian <= 0).sum()))
fig, axes = plt.subplots(1, 4, figsize=(12, 3), constrained_layout=True)
for ax, image, title in zip(axes, [fixed, moving, affine["warpedmovout"], warped],
                            ["Fixed", "Moving", "Affine", "Affine + B-spline SVF"]):
    ax.imshow(image[0, 0].cpu(), origin="lower", cmap="gray")
    ax.set_title(title); ax.axis("off")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3), constrained_layout=True)
stride = 3
gy, gx = np.mgrid[0:28:stride, 0:32:stride]
field = displacement[0].cpu().numpy()
axes[0].quiver(gx, gy, field[0, ::stride, ::stride], field[1, ::stride, ::stride],
               angles="xy", scale_units="xy", scale=1)
axes[0].set_title("Residual SVF displacement"); axes[0].set_aspect("equal")
view = axes[1].imshow(jacobian[0].cpu(), origin="lower", cmap="coolwarm")
axes[1].set_title("Total-map Jacobian")
fig.colorbar(view, ax=axes[1])
plt.show()

These short schedules and MSE are suitable for this synthetic same-contrast
pair. For anatomical data, choose physical spacing, similarity, regularization
and a multiresolution schedule deliberately; inspect inverse consistency,
boundary behavior and anatomical alignment in addition to the Jacobian.